Cell 1 — Imports and setup:

In [ ]:
import os
import cv2
import numpy as np
import mediapipe as mp
from tqdm.notebook import tqdm
import sys

# Add project root to path
sys.path.insert(0, os.path.abspath(".."))
import config

print("Libraries loaded successfully")
print("Device:", config.DEVICE)
print("Dataset path exists:", os.path.exists(config.KAGGLE_ASL_PATH))
print("Classes:", config.NUM_CLASSES)


Libraries loaded successfully
Device: cuda
Dataset path exists: True
Classes: 29


In [2]:
# Initialize MediaPipe hands
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=2,
    min_detection_confidence=0.5
)

def extract_landmarks(image_path):
    """Extract 126 hand landmark coordinates from an image"""
    image = cv2.imread(image_path)
    if image is None:
        return None
    
    # Convert BGR to RGB for MediaPipe
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = hands.process(image_rgb)
    
    landmarks = []
    if results.multi_hand_landmarks:
        for hand in results.multi_hand_landmarks:
            for lm in hand.landmark:
                landmarks.extend([lm.x, lm.y, lm.z])
    
    # Pad to 126 values if only one hand detected or no hand
    while len(landmarks) < 126:
        landmarks.append(0.0)
    
    return np.array(landmarks[:126], dtype=np.float32)

print("MediaPipe initialized")
print("Landmark extractor ready")
print("Output shape per image: 126 values (21 landmarks x 2 hands x 3 coords)")

MediaPipe initialized
Landmark extractor ready
Output shape per image: 126 values (21 landmarks x 2 hands x 3 coords)


In [3]:
# Test on one sample image before processing everything
sample_class = 'A'
sample_dir = os.path.join(config.KAGGLE_ASL_PATH, sample_class)
sample_image = os.listdir(sample_dir)[0]
sample_path = os.path.join(sample_dir, sample_image)

print(f"Testing on: {sample_image}")

# Extract landmarks
landmarks = extract_landmarks(sample_path)

if landmarks is not None:
    print(f"Landmarks extracted successfully")
    print(f"Shape: {landmarks.shape}")
    print(f"First 10 values: {landmarks[:10]}")
    print(f"Min value: {landmarks.min():.4f}")
    print(f"Max value: {landmarks.max():.4f}")
    print(f"\nTest PASSED - ready to process full dataset")
else:
    print("Test FAILED - no hand detected in sample image")
    print("Check your dataset path")

Testing on: 1.jpg
Landmarks extracted successfully
Shape: (126,)
First 10 values: [ 4.6490940e-01  7.0830953e-01 -5.0754363e-07  5.5521971e-01
  6.6132337e-01 -3.0121772e-02  6.1124432e-01  5.5698943e-01
 -3.7128199e-02  6.2857491e-01]
Min value: -0.0905
Max value: 0.7083

Test PASSED - ready to process full dataset


c:\Users\eng_b\miniconda3\envs\signify\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [4]:
# Full dataset preprocessing
os.makedirs(config.LANDMARKS_PATH, exist_ok=True)

classes = sorted([c for c in os.listdir(config.KAGGLE_ASL_PATH) 
                  if os.path.isdir(os.path.join(config.KAGGLE_ASL_PATH, c))])

print(f"Processing {len(classes)} classes...")
print(f"Saving landmarks to: {config.LANDMARKS_PATH}")
print("-" * 50)

total_saved = 0
total_failed = 0
class_summary = {}

for cls in classes:
    cls_path = os.path.join(config.KAGGLE_ASL_PATH, cls)
    images = [f for f in os.listdir(cls_path) 
              if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    landmarks_list = []
    failed = 0
    
    for img_file in tqdm(images, desc=f"{cls:10s}", leave=True):
        img_path = os.path.join(cls_path, img_file)
        lm = extract_landmarks(img_path)
        if lm is not None:
            landmarks_list.append(lm)
        else:
            failed += 1
    
    if landmarks_list:
        out_path = os.path.join(config.LANDMARKS_PATH, f"{cls}.npy")
        np.save(out_path, np.array(landmarks_list))
        total_saved += len(landmarks_list)
        total_failed += failed
        class_summary[cls] = {'saved': len(landmarks_list), 'failed': failed}
        print(f"  {cls:10s}: {len(landmarks_list):5d} saved, {failed:4d} failed")

print("-" * 50)
print(f"Total saved:  {total_saved:,}")
print(f"Total failed: {total_failed:,}")
print(f"Success rate: {total_saved/(total_saved+total_failed)*100:.1f}%")
print("\nPreprocessing complete")

Processing 29 classes...
Saving landmarks to: C:\Not Windows\Artificial Intelligence and Machine Learning (AIM)\Summer 2026\Capstone Project\data\processed\landmarks
--------------------------------------------------


A         :   0%|          | 0/8458 [00:00<?, ?it/s]

  A         :  8458 saved,    0 failed


B         :   0%|          | 0/8309 [00:00<?, ?it/s]

  B         :  8309 saved,    0 failed


...

  del       :  6836 saved,    0 failed


nothing   :   0%|          | 0/3030 [00:00<?, ?it/s]

  nothing   :  3030 saved,    0 failed


space     :   0%|          | 0/7071 [00:00<?, ?it/s]

  space     :  7071 saved,    0 failed
--------------------------------------------------
Total saved:  223,074
Total failed: 0
Success rate: 100.0%

Preprocessing complete


In [ ]:
import os
import numpy as np

print("Verifying saved landmark files...")
print("-" * 50)

total = 0
for cls in sorted(os.listdir(config.LANDMARKS_PATH)):
    if cls.endswith('.npy'):
        path = os.path.join(config.LANDMARKS_PATH, cls)
        data = np.load(path)
        total += len(data)
        print(f"  {cls.replace('.npy',''):10s}: {data.shape} — dtype: {data.dtype}")

print("-" * 50)
print(f"Total samples loaded: {total:,}")
print(f"Shape per sample: (126,) — 21 landmarks x 2 hands x 3 coords")

Verifying saved landmark files...
--------------------------------------------------
  A         : (8458, 126) — dtype: float32
  B         : (8309, 126) — dtype: float32
  C         : (8146, 126) — dtype: float32
  D         : (7629, 126) — dtype: float32
  E         : (7744, 126) — dtype: float32
  F         : (8031, 126) — dtype: float32
  G         : (7844, 126) — dtype: float32
  H         : (7906, 126) — dtype: float32
  I         : (7953, 126) — dtype: float32
  J         : (7503, 126) — dtype: float32
  K         : (7876, 126) — dtype: float32
  L         : (7939, 126) — dtype: float32
  M         : (7900, 126) — dtype: float32
  N         : (7932, 126) — dtype: float32
  O         : (8140, 126) — dtype: float32
  P         : (7601, 126) — dtype: float32
  Q         : (7954, 126) — dtype: float32
  R         : (8021, 126) — dtype: float32
  S         : (8109, 126) — dtype: float32
  T         : (8054, 126) — dtype: float32
  U         : (8023, 126) — dtype: float32
  V         